In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, min, max, datediff, to_date, date_trunc, expr, when
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder \
            .appName('Ecommerce-Analytic-Exploratory-Gold') \
            .master('local[*]') \
            .config('spark.driver.memory', '6g') \
            .config('spark.driver.executor', '6g') \
            .config('spark.shuffle.partitions', '200') \
            .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/01 20:08:42 WARN Utils: Your hostname, Liu, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/01 20:08:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/01 20:08:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
data_path = '../data/silver'

users = spark.read.parquet(f'{data_path}/users')
products = spark.read.parquet(f'{data_path}/products')
categories = spark.read.parquet(f'{data_path}/categories')
events = spark.read.parquet(f'{data_path}/events')

In [5]:
print('Users:')
users.printSchema()
print('Products:')
products.printSchema()
print('Categories:')
categories.printSchema()
print('Events:')
events.printSchema()

Users:
root
 |-- user_id: long (nullable = true)

Products:
root
 |-- product_id: long (nullable = true)
 |-- brand: string (nullable = true)
 |-- category_id: long (nullable = true)

Categories:
root
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_type: string (nullable = true)

Events:
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)
 |-- captured_price: double (nullable = true)



In [10]:
sales_trend_df = events.groupBy(to_date(col('event_time')).alias('event_date')) \
                        .agg(
                            sum(when(col('event_type') == 'view', 1).otherwise(0)).alias('total_views'),
                            sum(when(col('event_type') == 'cart', 1).otherwise(0)).alias('total_carts'),
                            sum(when(col('event_type') == 'purchase', 1).otherwise(0)).alias('total_purchases'),
                            sum(when(col('event_type') == 'purchase', col('captured_price')).otherwise(0.0)).alias('total_revenue')
                        ).orderBy('event_date')

sales_trend_df.show()
print(f'Total {sales_trend_df.count():,} rows')

+----------+-----------+-----------+---------------+--------------------+
|event_date|total_views|total_carts|total_purchases|       total_revenue|
+----------+-----------+-----------+---------------+--------------------+
|2019-11-01|    1116043|      16353|          19470|   5986270.709999872|
|2019-11-02|    1482273|      19475|          21903|  6421346.3999999575|
|2019-11-03|    1482434|      19640|          21393|   6374722.729999999|
|2019-11-04|    1772428|      22238|          27233|   8124310.640000036|
|2019-11-05|    1697628|      19380|          25146|   7360248.350000052|
|2019-11-06|    1631939|      19470|          24934|    7353034.05000007|
|2019-11-07|    1757701|      19815|          25142|   7147406.839999947|
|2019-11-08|    1768192|      69298|          25354|   7599813.770000043|
|2019-11-09|    1760692|      70441|          23072|   6683455.400000008|
|2019-11-10|    1857177|      72045|          22979|    6718664.85000002|
|2019-11-11|    1908236|      75139|  

Total 31 rows


In [11]:
brand_analysis_df = events.filter(col('event_type') == 'purchase') \
                            .join(products, 'product_id', 'inner') \
                            .groupby('brand') \
                            .agg(
                                count('product_id').alias('total_orders_sold'),
                                sum('captured_price').alias('total_brand_revenue')
                            ).orderBy(col('total_brand_revenue').desc())

brand_analysis_df.show()
print(f'Total {brand_analysis_df.count():,} rows')

+--------+-----------------+--------------------+
|   brand|total_orders_sold| total_brand_revenue|
+--------+-----------------+--------------------+
|   apple|           166076|1.2751733301000068E8|
| samsung|           200036| 5.487459584999878E7|
|    NULL|            90200|1.5804055949999997E7|
|  xiaomi|            68431|1.1292658200000027E7|
|      lg|            12884|   5241161.569999998|
|  huawei|            23726|   4798436.969999998|
|    sony|            10312|   3865243.809999992|
| lucente|            14563|  3528903.3900000015|
|    oppo|            15080|  3488540.7600000133|
|    acer|             6424|   3355824.690000002|
|  lenovo|             6637|  2722254.4599999986|
|   bosch|             8010|           1832717.5|
|    asus|             3077|  1676603.0300000005|
|   artel|             9267|  1642215.0399999998|
|      hp|             4117|  1349919.3899999994|
| indesit|             5196|          1310855.55|
|   haier|             3847|  1110829.0400000003|


Total 2,516 rows


In [ ]:


rfm_base = events.filter(col('event_type') == 'purchase') \
                .groupBy('user_id') \
                .agg(
                    max('event_time').alias('recency'),
                    count('user_session').alias('frequency'),
                    sum('captured_price').alias('monetary')
                ).orderBy('monetary')



2019-12-01 06:59:59
